In [ ]:
import pandas as pd
import os
from tqdm import tqdm
import copy
import re

In [ ]:
def extract_year(s):
    for part in s.split('/'):
        if re.match(r'\d{4}', part):
            return int(part)
    return s
        
def parse_ris(filename, json_mode: bool):
    if json_mode:
        format_regex = re.compile(r'(\w\w)  (.*)')
    else:
        format_regex = re.compile(r'(\w\w)  -(.*)')
    
    entry_template = {'year': None, 'authors': [], 'title': None, 'abstract': None, 'id': None, 'journal': None}
    entry = copy.deepcopy(entry_template)
    ris_entries = []
    error_example = None
    last_line_type = ''
    with open(filename, 'rt', encoding="utf-8") as f:
        for line in f.readlines():
            line = line.strip()
            if not line:
                if entry['year'] or entry['authors'] or entry['title']:
                    entry['authors'] = '\n'.join(entry['authors'])
                    ris_entries.append(entry)
                entry = copy.deepcopy(entry_template)
                last_line_type = ''
                continue

            match = format_regex.fullmatch(line)
            if not match:
                if last_line_type == 'AB':
                    entry['abstract'] += '\n' + line
                    continue
                if last_line_type in ('KW', 'N1', 'AD'):
                    continue
                error_example = line
                # print(line)
                continue
                
            typ, text = match.group(1), match.group(2)
            text = text.strip()
            if typ in ('TI','T1'):
                entry['title'] = text
            elif typ == 'Y1':
                entry['year'] = extract_year(text)
            elif typ == 'AU':
                entry['authors'].append(text)
            elif typ == 'AB':
                entry['abstract'] = text
            elif typ == 'SN':
                entry['id'] = text
            elif typ in ('T2', 'JO', 'JF', 'J1'):
                entry['journal'] = text

            last_line_type = typ
    
        if entry['year'] or entry['authors'] or entry['title']:
            entry['authors'] = '\n'.join(entry['authors'])
            ris_entries.append(entry)
    
    return pd.DataFrame(ris_entries), error_example
    

In [ ]:
file_list = []
dir1 = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\Paul\SR\SR\RIS'
for filename in os.listdir(dir1):
    file_list.append(os.path.join(dir1, filename))
dir2 = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\Paul\SR\SR\uploaded studies txt'
for filename in os.listdir(dir2):
    file_list.append(os.path.join(dir2, filename))
file_list

In [ ]:
parts = []
for filename in tqdm(file_list):
    if filename.endswith('.nbib'):
        continue
    df, error_example = parse_ris(filename, filename.endswith('.json'))
    if error_example:
        print(f'Errors in {filename}')
        print(error_example)
    df['filename'] = os.path.basename(filename)
    parts.append(df)
big_ris_df = pd.concat(parts)
print(len(big_ris_df))
big_ris_df.head()

In [ ]:
pubmed_df = pd.read_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\549_results_from_PUBMED.xlsx')
print(pubmed_df.columns)

In [ ]:
pubmed_df['filename'] = 'pubmed-longcovidO-set.nbib'
pubmed_df.head()

In [ ]:
pubmed_df.rename(columns={'Title': 'title', 'Authors': 'authors', 'Journal/Book': 'journal', 'Publication Year': 'year'}, inplace=True)
pubmed_df.head()

In [ ]:
combined_df = pd.concat([big_ris_df, pubmed_df], axis=0)
combined_df.head()

In [ ]:
def extract_letters(s):
    s = re.sub(r'\W', '', s)
    return s.lower()

extract_letters('What’s going on following acute covid-19? Clinical characteristics of patients in an out-patient rehabilitation program')

In [ ]:
paul_df = pd.read_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\paul\COVID19_Studies_Papers_Paul.xlsx')
paul_df['title_letters'] = paul_df['Title'].apply(extract_letters)

In [ ]:
combined_df['title_year'] =  combined_df.apply(lambda row: f'{row["title"]}_{row["year"]}', axis=1)
combined_df['title_letters'] = combined_df['title'].apply(extract_letters)

In [ ]:
combined_df['is_in_paul'] = combined_df.title_letters.isin(paul_df['title_letters'])

In [ ]:
combined_df.to_excel('joined_sources.xlsx', index=False)

In [ ]:
paul_df['is_in_others'] = paul_df.title_letters.isin(combined_df.title_letters)
len(paul_df), paul_df.is_in_others.sum()

In [ ]:
paul_df.to_excel('paul_df_with_others.xlsx', index=False)

In [ ]:
combined_df.title.nunique()

In [ ]:
Perceived Care and Well-being of Patients With Cancer and Matched Norm Participants in the COVID-19 Crisis: Results of a Survey of Participants in the Dutch PROFILES Registry.
Perceived Care and Well-being of Patients With Cancer and Matched Norm Participants in the COVID-19 Crisis: Results of a Survey of Participants in the Dutch PROFILES Registry.
